# Chapter 4: The Operational Layer

This notebook sets up the Lakebase (managed Postgres) operational layer for the vehicle tracking system. We create three tables -- `vehicles`, `vehicle_positions` and `trips` -- that hold the live transactional data written by the simulator in Chapter 5 and read by the Streamlit app in Chapter 6.

## 1. Install Dependencies

In [ ]:
%pip install psycopg2-binary==2.9.12 \
             pyyaml==6.0.3 --quiet

print("Install complete.")

## 2. Imports

In [ ]:
import os
import psycopg2

from config_validator import load_config, ConfigError

## 3. Configuration

We're using a local Postgres installation as a temporary stand-in for Lakebase
while a Databricks free tier provisioning issue is resolved. The connection
details and all table definitions are identical -- swapping back to Lakebase
requires only updating the connection parameters in this section and Section 4.

On a Homebrew Mac install, Postgres creates a role matching the Mac username
rather than a `postgres` superuser. We use `os.environ.get('USER')` to pick
this up automatically.

In [ ]:
PG_HOST   = "localhost"
PG_PORT   = 5432
PG_DBNAME = "postgres"
PG_USER   = os.environ.get("USER")

try:
    cfg = load_config("config.yaml")
except (FileNotFoundError, ConfigError) as e:
    raise SystemExit(f"Config error: {e}")

print(f"City: {cfg['city']['name']}")
print("Configuration set.")

## 4. Connect to Local Postgres

In [ ]:
conn = psycopg2.connect(
    host   = PG_HOST,
    port   = PG_PORT,
    dbname = PG_DBNAME,
    user   = PG_USER
)
conn.autocommit = True
cursor = conn.cursor()

print("Connected to local Postgres.")

## 5. Drop Existing Tables

In [ ]:
cursor.execute("DROP TABLE IF EXISTS trips CASCADE")
cursor.execute("DROP TABLE IF EXISTS vehicle_positions CASCADE")
cursor.execute("DROP TABLE IF EXISTS vehicles CASCADE")

print("Existing tables dropped.")

## 6. Create Tables

In [ ]:
# vehicles -- one row per vehicle, slow-changing
cursor.execute("""
    CREATE TABLE vehicles (
        vehicle_id  TEXT PRIMARY KEY,
        driver_name TEXT NOT NULL,
        zone        TEXT NOT NULL,
        status      TEXT NOT NULL DEFAULT 'idle',
        created_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
    )
""")

# vehicle_positions -- high-frequency inserts, one row per position update
cursor.execute("""
    CREATE TABLE vehicle_positions (
        position_id  BIGSERIAL PRIMARY KEY,
        vehicle_id   TEXT NOT NULL REFERENCES vehicles(vehicle_id),
        lat          DOUBLE PRECISION NOT NULL,
        lon          DOUBLE PRECISION NOT NULL,
        speed_kmh    DOUBLE PRECISION,
        current_zone TEXT,
        recorded_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
    )
""")

# trips -- one row per trip, tracks lifecycle from request to completion
cursor.execute("""
    CREATE TABLE trips (
        trip_id      TEXT PRIMARY KEY,
        vehicle_id   TEXT REFERENCES vehicles(vehicle_id),
        pickup_lat   DOUBLE PRECISION NOT NULL,
        pickup_lon   DOUBLE PRECISION NOT NULL,
        dropoff_lat  DOUBLE PRECISION NOT NULL,
        dropoff_lon  DOUBLE PRECISION NOT NULL,
        pickup_zone  TEXT,
        dropoff_zone TEXT,
        status       TEXT NOT NULL DEFAULT 'requested',
        requested_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
        started_at   TIMESTAMPTZ,
        completed_at TIMESTAMPTZ
    )
""")

print("Tables created.")

## 7. Create Indexes

In [ ]:
# Fast lookup of latest positions per vehicle
cursor.execute("""
    CREATE INDEX idx_vehicle_positions_vehicle_id
    ON vehicle_positions (vehicle_id, recorded_at DESC)
""")

# Fast lookup of vehicles by status and zone (used by nearest-driver query)
cursor.execute("""
    CREATE INDEX idx_vehicles_status_zone
    ON vehicles (status, zone)
""")

# Fast lookup of trips by status
cursor.execute("""
    CREATE INDEX idx_trips_status
    ON trips (status)
""")

print("Indexes created.")

## 8. Seed Vehicles

In [ ]:
# Vehicles spread across the zones
vehicles = [
    (v["id"], v["driver"], v["zone"])
    for v in cfg["vehicles"]
]

cursor.executemany("""
    INSERT INTO vehicles (vehicle_id, driver_name, zone)
    VALUES (%s, %s, %s)
""", vehicles)

print(f"Seeded {len(vehicles)} vehicles.")

## 9. Verify

In [ ]:
# Table row counts
for table in ["vehicles", "vehicle_positions", "trips"]:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"{table:<20} {count:,} rows")

print()

# Vehicles by zone
cursor.execute("""
    SELECT zone, COUNT(*) AS vehicles
    FROM vehicles
    GROUP BY zone
    ORDER BY zone
""")
print("Vehicles per zone:")
for row in cursor.fetchall():
    print(f"  {row[0]:<15} {row[1]}")

print()

# Sample vehicle records
cursor.execute("SELECT vehicle_id, driver_name, zone, status FROM vehicles ORDER BY vehicle_id LIMIT 5")
print("Sample vehicles:")
print(f"  {'ID':<8} {'Driver':<18} {'Zone':<15} {'Status'}")
for row in cursor.fetchall():
    print(f"  {row[0]:<8} {row[1]:<18} {row[2]:<15} {row[3]}")

## 10. Teardown

In [ ]:
cursor.close()
conn.close()
print("Connection closed.")